# Análisis EDA y Clasificación de Tickets de Incidentes Bancarios

Este notebook realiza un análisis exploratorio de datos (EDA) y un modelo de clasificación para predecir la categoría de cada ticket de incidentes bancarios a partir de un archivo CSV.

## 1. Carga de Librerías y Configuración Inicial

In [ ]:
# ── Librería estándar ─────────────────────────────────────────────────────────
import os
import re
import unicodedata
import warnings
from collections import Counter

# ── Datos y visualización ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# ── Procesamiento de texto ────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords

# ── Estadística y matrices dispersas ─────────────────────────────────────────
from scipy.stats import f_oneway
from scipy.sparse import vstack

# ── Scikit-learn: features y preprocesamiento ────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Scikit-learn: modelos ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# ── Scikit-learn: selección de modelos y métricas ────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)

# ── Imbalanced-learn ─────────────────────────────────────────────────────────
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings('ignore')

# ── Configuración de gráficos ─────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(rc={'figure.figsize': (10, 6)})


## 2. Carga y Exploración Inicial de Datos

In [ ]:
# Cargar el archivo CSV
# Ruta relativa desde Analisis/ hacia Data/ (un nivel arriba)
ruta_csv = "../Datos/IncidentesCategorizados_v2.csv"
print('Directorio actual:', os.getcwd())
print('Ruta CSV:', ruta_csv)

# Intenta cargar el archivo
try:
    try:
        df = pd.read_csv(ruta_csv, sep=",", encoding="latin-1")
        if len(df.columns) == 1:
            raise ValueError("Wrong delimiter")
    except (ValueError, pd.errors.ParserError):
        df = pd.read_csv(ruta_csv, sep=";", encoding="latin-1")
except FileNotFoundError as e:
    print('No se encontró el archivo. Verifica la ruta y ubicación.')
    raise e

# Mostrar primeras filas y resumen
print('Dimensiones:', df.shape)
df.head()


In [ ]:
# Cominando cmdb_ci_business_app y cmdb_ci dado que corresponden al mismo campo
# 1. Asegurar que cadenas vacías o espacios se interpreten como NaN
df['cmdb_ci_business_app'] = df['cmdb_ci_business_app'].replace(r'^\s*$', np.nan, regex=True)
df['cmdb_ci'] = df['cmdb_ci'].replace(r'^\s*$', np.nan, regex=True)

# 2. Rellenar los valores nulos de cmdb_ci_business_app con los de cmdb_ci
df['cmdb_ci_business_app'] = df['cmdb_ci_business_app'].combine_first(df['cmdb_ci'])

# 3. Eliminar la columna antigua redundante
df = df.drop(columns=['cmdb_ci'])

In [ ]:
# Tipos de datos y valores nulos
df.info()
df.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
# Exploración rápida de variables informativas
print('\nPorcentaje de valores nulos por columna:')
print((df.isnull().mean()*100).sort_values(ascending=False))
print('\nNúmero de valores únicos por columna:')
print(df.nunique().sort_values(ascending=False))

# Sugerencia de variables informativas (no ID, no columnas vacías o con un solo valor)
columnas_informativas = [col for col in df.columns if df[col].nunique()>1 and df[col].nunique()<df.shape[0]*0.9 and not col.lower().startswith('id')]
print('\nColumnas potencialmente informativas:')
print(columnas_informativas)


## 3. Limpieza y Preprocesamiento de Datos

## 3.1 Columna Clasificación

In [ ]:
# Cambiar el nombre de la columna 'Categoria' por 'Clasificación'
df = df.rename(columns={"Categoría": "Clasificación"})

In [ ]:
# Mostrar la distribución de clases (conteo de tickets por clasificación)
conteo_clases = df['Clasificación'].value_counts(dropna=False)
print(conteo_clases)

In [ ]:
# Revisar valores únicos y nulos en 'Clasificación'
if 'Clasificación' in df.columns:
    print('Valores únicos en Clasificación:', df['Clasificación'].unique())
    print('Valores nulos en Clasificación:', df['Clasificación'].isnull().sum())
else:
    print("La columna 'Clasificación' no está presente en el DataFrame.")

In [ ]:
# Normalización de valores en la columna 'Clasificación'
def normalizar_texto(texto):
    if pd.isnull(texto):
        return "sin_clasificar"
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.replace('.', '')
    texto = texto.replace('/', '-')
    texto = texto.strip()
    return texto

df['Clasificación'] = df['Clasificación'].apply(normalizar_texto)

# Verifica los valores únicos después de la normalización
# Revisar valores únicos y nulos en 'Clasificación'
if 'Clasificación' in df.columns:
    print('Valores únicos en Clasificación:', df['Clasificación'].unique())
    print('Valores nulos en Clasificación:', df['Clasificación'].isnull().sum())
else:
    print("La columna 'Clasificación' no está presente en el DataFrame.")


In [ ]:
# 1. Agrupar valores en plural y singular en la columna 'Clasificación'
agrupaciones_plural = {
    'cheques': 'cheque',
    'cuentas': 'cuenta',
    'activos': 'activo',
    'garantias': 'garantia',
    'inversiones': 'inversion',
    'sobregiros': 'sobregiro',
    'usuarios': 'usuario',
    'procesos': 'proceso',
    'clientes': 'cliente',
    'reportes online': 'reporte online',
    'reportes batch': 'reporte batch',
    'formas numeradas': 'forma numerada',
    # Agregar más se detectan otros casos
}
df['Clasificación'] = df['Clasificación'].replace(agrupaciones_plural)

In [ ]:
# Reemplazar valores sin clasificar y pendientes → sin_clasificar
df['Clasificación'] = df['Clasificación'].replace(
#    ['0', 0, None, np.nan, 'no aplica', 'pendiente'],
    ['0', 0, None, np.nan, 'pendiente'],
    'sin_clasificar'
)

# Eliminar filas con clases que tienen muy pocos ejemplos para entrenar (< 3) o son irrelevantes
clases_a_eliminar = ['cc-cancelacion batch', 'cc-evento-alerta', 'turno']
filas_antes = len(df)
df = df[~df['Clasificación'].isin(clases_a_eliminar)].reset_index(drop=True)
print(f"Filas eliminadas: {filas_antes - len(df)}  ({clases_a_eliminar})")
print(f"Total filas restantes: {len(df)}")


In [ ]:
# Verifica los valores únicos después de la normalización
print(df['Clasificación'].unique())


In [ ]:
# Mostrar la distribución de clases (conteo de tickets por clasificación)
conteo_clases = df['Clasificación'].value_counts(dropna=False)
print(f"Total de clases de Clasificación: {conteo_clases.shape[0]}")
print()
print(conteo_clases)


In [ ]:
# Verificar duplicados en el DataFrame
duplicados = df[df.duplicated()]
print(f"Número de filas duplicadas: {duplicados.shape[0]}")
if not duplicados.empty:
    display(duplicados.head())
else:
    print("No se encontraron filas duplicadas.")

## 4. Análisis Exploratorio de Datos (EDA)

In [ ]:
# ==============================================================================
# ANÁLISIS INTEGRAL DE RELEVANCIA DE VARIABLES PARA LA CLASIFICACIÓN DE TICKETS
# ==============================================================================
# Evalúa todas las columnas del conjunto de datos para identificar cuáles aportan
# poder predictivo real y cuáles deben descartarse (por varianza nula o data leakage).

from scipy.stats import chi2_contingency

# 1. Filtrar registros etiquetados válidos para medir asociación real
mask_valido = ~df['Clasificación'].astype(str).str.strip().str.lower().isin(
    ['0', 'nan', 'sin_clasificar', 'pendiente', 'none', '']
) & df['Clasificación'].notnull()

df_eval = df[mask_valido].copy()
print(f"Evaluando relevancia predictiva sobre {len(df_eval)} tickets clasificados ({df_eval['Clasificación'].nunique()} categorías).\n")

# Función para calcular V de Cramér con corrección de sesgo (Bergsma, 2013)
def cramers_v_corrected(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] <= 1 or confusion_matrix.shape[1] <= 1:
        return 0.0
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2corr / denom))

# 2. Clasificación funcional de columnas según el ciclo de vida del ticket
columnas_fuga = ['close_notes', 'resolved_at', 'business_duration', 'reopen_count', 'state', 'assigned_to']
columnas_id = ['number', 'sys_created_on', 'caller_id', 'u_affected_user']
columnas_texto = ['short_description', 'description', 'comments_and_work_notes']

columnas_categoricas = [
    col for col in df.columns 
    if col not in columnas_fuga + columnas_id + columnas_texto + ['Clasificación']
    and df[col].nunique() > 1
]

# 3. Evaluación de variables categóricas candidatas
resultados = []
for col in columnas_categoricas:
    s_full = df[col]
    s_eval = df_eval[col].fillna('DESCONOCIDO').astype(str)
    
    nunique = s_full.nunique(dropna=False)
    pct_null = s_full.isnull().mean() * 100
    top_val_pct = (s_full.value_counts(normalize=True, dropna=False).iloc[0]) * 100 if len(s_full) > 0 else 0
    
    cv = cramers_v_corrected(s_eval, df_eval['Clasificación'])
    
    # Diagnóstico y utilidad
    if top_val_pct >= 95.0:
        utilidad = "Descartar"
        motivo = "Cuasi-constante (Varianza nula)"
    elif cv >= 0.35:
        utilidad = "Alta"
        motivo = "Fuerte asociación con Clasificación"
    elif cv >= 0.20:
        utilidad = "Media"
        motivo = "Moderada asociación discriminante"
    else:
        utilidad = "Baja"
        motivo = "Poca capacidad discriminante"
        
    resultados.append({
        'Variable': col,
        'Nulos (%)': round(pct_null, 1),
        'Valores Únicos': nunique,
        'Dominancia Moda (%)': round(top_val_pct, 1),
        "Cramér's V": round(cv, 4),
        'Utilidad': utilidad,
        'Diagnóstico': motivo
    })

df_ranking = pd.DataFrame(resultados).sort_values(by="Cramér's V", ascending=False).reset_index(drop=True)

print("=" * 85)
print("1. RANKING DE RELEVANCIA ESTADÍSTICA (VARIABLES CATEGÓRICAS)")
print("=" * 85)
print(df_ranking[['Variable', 'Nulos (%)', 'Dominancia Moda (%)', "Cramér's V", 'Utilidad', 'Diagnóstico']].to_string(index=False))

# 4. Diagnóstico de campos de texto (NLP)
print("\n" + "=" * 85)
print("2. CAMPOS DE TEXTO (PRINCIPAL FUENTE DE INFORMACIÓN PARA EL CLASIFICADOR)")
print("=" * 85)
for col in columnas_texto:
    if col in df.columns:
        pct_nulos = df[col].isnull().mean() * 100
        avg_len = df[col].fillna('').astype(str).str.len().mean()
        estado = "RECOMENDADA (Core del modelo NLP)" if pct_nulos < 10 else "Opcional (Muchos nulos / Notas internas)"
        print(f"• {col:<25}: {pct_nulos:5.1f}% nulos | Longitud media: {avg_len:5.1f} caracteres -> {estado}")

# 5. Advertencia de Fuga de Información
print("\n" + "=" * 85)
print("3. VARIABLES EXCLUIDAS POR FUGA DE DATOS (DATA LEAKAGE)")
print("=" * 85)
print("Las siguientes columnas NO deben usarse si el modelo clasifica tickets al crearse (estado 'Nuevo'):")
for col in columnas_fuga:
    if col in df.columns:
        print(f"⚠ {col:<25}: Se genera o modifica durante/después de la gestión del incidente.")

# 6. Gráficos de Alto Valor
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico A: Barras de Cramér's V ordenadas por utilidad
paleta = {'Alta': '#2ca02c', 'Media': '#ff7f0e', 'Descartar': '#d62728', 'Baja': '#7f7f7f'}
sns.barplot(
    data=df_ranking,
    x="Cramér's V",
    y='Variable',
    hue='Utilidad',
    palette=paleta,
    dodge=False,
    ax=axes[0]
)
axes[0].set_title("Poder Discriminante de Variables Categóricas (V de Cramér)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("V de Cramér (0 = Sin relación, 1 = Relación perfecta)")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(title="Utilidad")

# Gráfico B: Proporción de Clases en la Variable Categórica Top (cmdb_ci_business_app)
top_col = 'cmdb_ci_business_app' if 'cmdb_ci_business_app' in df_eval.columns else df_ranking.iloc[0]['Variable']
top_categorias = df_eval['Clasificación'].value_counts().head(8).index
df_top_clases = df_eval[df_eval['Clasificación'].isin(top_categorias)]

# Frecuencias relativas (% por fila) para ver la mezcla de categorías en cada aplicación
top_apps = df_top_clases[top_col].value_counts().head(8).index
crosstab_norm = pd.crosstab(
    df_top_clases[df_top_clases[top_col].isin(top_apps)][top_col],
    df_top_clases['Clasificación'],
    normalize='index'
) * 100

sns.heatmap(crosstab_norm, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': '% de la aplicación'}, ax=axes[1])
axes[1].set_title(f"Distribución Relativa (% fila): '{top_col}' vs Top Clases", fontsize=12, fontweight='bold')
axes[1].set_ylabel(top_col)
axes[1].set_xlabel("Clasificación")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 5. Preparación de Datos para Modelado

In [ ]:
# Cargar stopwords ANTES de limpiar_texto para poder usarlas en la limpieza
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))
print(f"✓ Stopwords cargadas: {len(spanish_stopwords)} palabras")


In [ ]:
# Unificar texto relevante para análisis de similitud y modelado NLP
# Enriquecemos el texto con los metadatos de alto poder predictivo identificados en el EDA:
# - cmdb_ci_business_app (aplicación bancaria consolidada con cmdb_ci)
# - u_subcategory y u_subcategory_2 (subcategorías técnicas del incidente)
# - u_affected_user.title (cargo o rol del usuario solicitante)
# - u_affected_user.company (empresa: Banco Pichincha vs Tata Consultancy Services)
# - u_affected_user.department (departamento o área del usuario)

df['texto_unificado'] = (
    df['short_description'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['cmdb_ci_business_app'].fillna('') + ' ' +
    df['u_subcategory'].fillna('') + ' ' +
    df['u_subcategory_2'].fillna('') + ' ' +
    df['u_affected_user.title'].fillna('') + ' ' +
    df['u_affected_user.company'].fillna('') + ' ' +
    df['u_affected_user.department'].fillna('')
)

# Guardar copia del texto crudo ANTES de limpiar (para análisis de duplicados y GroupSplit)
df['texto_unificado_raw'] = df['texto_unificado']

print(f"✓ 'texto_unificado' generado con éxito para {len(df)} registros.")
print(f"  Longitud promedio del texto enriquecido: {df['texto_unificado'].str.len().mean():.1f} caracteres.")



In [ ]:
# ── Análisis de duplicados exactos en texto_unificado (PRE-limpieza) ──────────
# Usa texto_unificado_raw para que sea robusto sin importar el orden de ejecución.

dup_mask_raw = df['texto_unificado_raw'].duplicated(keep=False)
df_dups_raw = df[dup_mask_raw].copy()

print(f"Tickets con texto_unificado duplicado (crudo): {dup_mask_raw.sum()}  "
      f"({dup_mask_raw.mean():.1%} del total)")
print(f"Textos únicos que se repiten: {df_dups_raw['texto_unificado_raw'].nunique()}")

# Tabla de frecuencias: cuántas veces se repite cada texto y sus clasificaciones
resumen_dups = (
    df_dups_raw
    .groupby('texto_unificado_raw', as_index=False)
    .agg(
        n_tickets       = ('number', 'count'),
        clasificaciones = ('Clasificación', lambda x: ', '.join(sorted(x.unique()))),
        n_clases        = ('Clasificación', 'nunique'),
        ejemplos        = ('number', lambda x: ', '.join(x.head(4).astype(str)))
    )
    .rename(columns={'texto_unificado_raw': 'texto_unificado'})
    .sort_values('n_tickets', ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 textos más repetidos:")
with pd.option_context('display.max_colwidth', 80):
    display(resumen_dups.head(20))

# Duplicados con clasificaciones distintas (ambigüedad real)
ambiguos = resumen_dups[resumen_dups['n_clases'] > 1]
print(f"\nTextos duplicados con DISTINTAS clasificaciones (ambigüedad): {len(ambiguos)}")
if len(ambiguos):
    with pd.option_context('display.max_colwidth', 80):
        display(ambiguos.head(20))


In [ ]:
# ── IDs de tickets ambiguos (ninguna clasificación es "sin_clasificar") ───────
textos_ambiguos_reales = resumen_dups[
    (resumen_dups['n_clases'] > 1) &
    (~resumen_dups['clasificaciones'].str.contains(r'\bsin_clasificar\b', na=False))
]['texto_unificado'].tolist()

ids_ambiguos = (
    df[df['texto_unificado_raw'].isin(textos_ambiguos_reales)]['number']
    .sort_values()
    .reset_index(drop=True)
)

print(f"Total IDs para revisión: {len(ids_ambiguos)}")
display(ids_ambiguos.to_frame())


In [ ]:
# ── Contenido completo de los tickets ambiguos para revisión ──────────────────
cols_mostrar = ['number', 'texto_unificado_raw', 'Clasificación']
# Agrega columnas extra si existen en df
for col_extra in ['short_description', 'description', 'Subcategoría', 'category', 'subcategory']:
    if col_extra in df.columns and col_extra not in cols_mostrar:
        cols_mostrar.append(col_extra)

df_revision = (
    df[df['number'].isin(ids_ambiguos)]
    [cols_mostrar]
    .sort_values('number')
    .reset_index(drop=True)
)

print(f"Tickets para revisión: {len(df_revision)}")
with pd.option_context('display.max_colwidth', 200, 'display.max_rows', None):
    display(df_revision)


In [ ]:
mapa_mayoritario = {}
for texto in textos_ambiguos_reales:
    conteo = (
        df[df['texto_unificado_raw'] == texto]['Clasificación']
        .value_counts()
    )
    mapa_mayoritario[texto] = conteo.idxmax()

mask_ambiguos = df['texto_unificado_raw'].isin(textos_ambiguos_reales)
df.loc[mask_ambiguos, 'Clasificación'] = df.loc[mask_ambiguos, 'texto_unificado_raw'].map(mapa_mayoritario)

print(f"Tickets corregidos: {mask_ambiguos.sum()}")
print("\nClasificación asignada por texto ambiguo:")
for texto, clase in mapa_mayoritario.items():
    print(f" → '{texto[:80]}...' ► {clase}")

verificacion = (
    df[mask_ambiguos]
    .groupby('texto_unificado_raw')['Clasificación']
    .nunique()
)
assert (verificacion == 1).all(), "¡Aún quedan textos con múltiples clases!"
print(f"\n✓ Verificación OK: todos los textos ambiguos tienen ahora una única clasificación.")


In [ ]:
# ── Preservación de registros y preparación para Group Split ────────────────
# Tras resolver ambigüedades con la clase mayoritaria, se conservan TODOS los registros.
# Las plantillas y textos recurrentes representan la distribución real de incidentes del banco.
# Para evitar Data Leakage (que un mismo texto esté en Train y en Test),
# la separación se realizará mediante GroupShuffleSplit agrupando por 'texto_unificado_raw'.

total_tickets = len(df)
textos_unicos = df['texto_unificado_raw'].nunique()
tickets_repetidos = total_tickets - textos_unicos

print(f"Total de tickets conservados : {total_tickets} (100% de la información preservada)")
print(f"Textos únicos de incidentes  : {textos_unicos}")
print(f"Tickets con textos recurrentes: {tickets_repetidos} ({tickets_repetidos / total_tickets:.1%})")
print(f"-> Nota: No se descarta ningún ticket. El aislamiento train/test se garantizará mediante GroupShuffleSplit.")

print(f"\nDistribución de clases (conteo total de incidentes):")
print(df['Clasificación'].value_counts().to_string())


In [ ]:
def limpiar_texto(texto):
    if pd.isnull(texto):
        return ""
    texto = str(texto).lower()
    # Normalizar acentos y caracteres especiales
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    # Eliminar caracteres no alfanuméricos (excepto espacios)
    texto = re.sub(r'[^a-zA-Z0-9\s]', ' ', texto)
    # Eliminar tokens puramente numéricos (códigos, fechas, horas, IDs)
    texto = re.sub(r'\b\d+\b', ' ', texto)
    # Eliminar tokens alfanuméricos que sean mayormente números (ej: ws0067, v1, v2)
    texto = re.sub(r'\b[a-z]*\d+[a-z0-9]*\b', ' ', texto)
    # Eliminar tokens muy cortos (1-2 caracteres) que no aportan significado
    texto = re.sub(r'\b\w{1,2}\b', ' ', texto)
    # Normalizar espacios
    texto = re.sub(r'\s+', ' ', texto).strip()
    # Eliminar stopwords en español directamente en el texto
    tokens = texto.split()
    tokens = [t for t in tokens if t not in spanish_stopwords]
    return ' '.join(tokens)

df['texto_unificado'] = df['texto_unificado'].apply(limpiar_texto)

# Verificar mejora — mostrar muestra de textos limpios
print("Ejemplos de texto limpio (sin stopwords):")
for t in df['texto_unificado'].dropna().sample(5, random_state=42):
    print(" →", t[:120])


In [ ]:
# Separar tickets clasificados y no clasificados
df_clasificados = df[df['Clasificación'] != 'sin_clasificar'].copy()
df_no_clasificados = df[df['Clasificación'] == 'sin_clasificar'].copy()
print(f"Tickets clasificados: {df_clasificados.shape[0]}")
print(f"Tickets sin clasificar: {df_no_clasificados.shape[0]}")

## Vectorización 

In [ ]:
# ===== PASO 1: VECTORIZACIÓN ÚNICA Y CONSISTENTE =====
# Las stopwords ya fueron eliminadas en limpiar_texto → no se repasan aquí
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Vectorizar datos clasificados
X_clasificados = vectorizer.fit_transform(df_clasificados['texto_unificado'])
y_clasificados = df_clasificados['Clasificación'].values

# Vectorizar datos sin clasificar (usando el MISMO vectorizer)
X_no_clasificados = vectorizer.transform(df_no_clasificados['texto_unificado'])

print("Distribución original de clases:", Counter(y_clasificados))
print(f"Matriz X_clasificados shape: {X_clasificados.shape}")
print(f"Vectorizer features: {vectorizer.get_feature_names_out().shape[0]}")


In [ ]:
features = vectorizer.get_feature_names_out()
print(f"Total features: {len(features)}")
print("Primeras 20:", features[:20])
print("Últimas 20:", features[-20:])

## Balanceo de clases

In [ ]:
# ===== PASO 2: PARTICIÓN POR GRUPOS (GroupShuffleSplit) =====
# Se agrupan los tickets por su 'texto_unificado_raw' para garantizar que ningún
# texto idéntico o plantilla compartida aparezca simultáneamente en Train y Test.
# Esto previene el Data Leakage y evalúa la capacidad de generalización real sin destruir datos.

from sklearn.model_selection import GroupShuffleSplit

# Asegurar tipos homogéneos de etiquetas
y_clasificados_str = np.array(y_clasificados, dtype=str)
grupos = df_clasificados['texto_unificado_raw'].astype(str).values

# Clases con un solo grupo único van directo a Train para asegurar que el modelo las conozca
grupos_por_clase = df_clasificados.groupby('Clasificación')['texto_unificado_raw'].nunique()
clases_un_solo_grupo = grupos_por_clase[grupos_por_clase < 2].index.tolist()

indices = np.arange(len(df_clasificados))

if clases_un_solo_grupo:
    print(f"  Clases con 1 solo grupo único (van directo a Train): {clases_un_solo_grupo}")
    mask_un_grupo = df_clasificados['Clasificación'].isin(clases_un_solo_grupo).values
    idx_un_grupo = indices[mask_un_grupo]
    idx_multigrupo = indices[~mask_un_grupo]

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_rel, test_rel = next(
        gss.split(
            X_clasificados[idx_multigrupo],
            y_clasificados_str[idx_multigrupo],
            groups=grupos[idx_multigrupo]
        )
    )
    idx_train = np.concatenate([idx_multigrupo[train_rel], idx_un_grupo])
    idx_test = idx_multigrupo[test_rel]
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_rel, test_rel = next(
        gss.split(
            X_clasificados,
            y_clasificados_str,
            groups=grupos
        )
    )
    idx_train = indices[train_rel]
    idx_test = indices[test_rel]

# Generar matrices y etiquetas de Train y Test
X_train, y_train = X_clasificados[idx_train], y_clasificados_str[idx_train]
X_test, y_test = X_clasificados[idx_test], y_clasificados_str[idx_test]

# Reconstruir DataFrame base del test set para auditoría posterior
df_test_base = df_clasificados.iloc[idx_test].copy().reset_index(drop=True)
df_test_base['y_real'] = y_test

# Verificación estricta de cero fuga de datos
overlap_grupos = set(grupos[idx_train]).intersection(set(grupos[idx_test]))
assert len(overlap_grupos) == 0, f"¡Alerta! Hay {len(overlap_grupos)} grupos compartidos."

print(f"✓ Train set: {X_train.shape[0]} muestras ({X_train.shape[0] / len(df_clasificados):.1%})")
print(f"✓ Test set:  {X_test.shape[0]} muestras ({X_test.shape[0] / len(df_clasificados):.1%})")
print(f"✓ Total grupos únicos Train: {len(set(grupos[idx_train]))} | Test: {len(set(grupos[idx_test]))}")
print(f"✓ Overlap de textos entre Train y Test: {len(overlap_grupos)} (Data Leakage = 0)")
print(f"✓ df_test_base: {df_test_base.shape[0]} filas listas para evaluación")

In [ ]:
# ===== PASO 3: OVERSAMPLING SOLO EN TRAIN =====
# Se usa RandomOverSampler en lugar de SMOTE porque:
#   - SMOTE interpola entre vecinos → genera vectores TF-IDF promedio sin sentido lingüístico
#   - RandomOverSampler duplica tickets reales → mantiene la semántica del texto original
#   - Para datos de texto dispersos (sparse), ROS es la opción más adecuada

ros = RandomOverSampler(random_state=42)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train, y_train)

conteo_antes  = Counter(y_train)
conteo_despues = Counter(y_train_balanced)

print(f"✓ Balanceo con RandomOverSampler")
print(f"  Muestras antes:   {X_train.shape[0]}")
print(f"  Muestras después: {X_train_balanced.shape[0]}")
print(f"\nDistribución después del balanceo ({len(conteo_despues)} clases, todas igualadas):")
print(conteo_despues)


## KNN

### Método del codo

In [ ]:
# ===== PASO 4: BÚSQUEDA DE k ÓPTIMO CON CROSS-VALIDATION =====
# Se evalúa con StratifiedKFold sobre X_train_balanced — X_test NO se toca aquí.
# Cada fold actúa como validación interna; el test set queda reservado para evaluación final.

k_range = range(1, 16)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    fold_f1 = []
    for train_idx, val_idx in cv.split(X_train_balanced, y_train_balanced):
        X_fold_train, y_fold_train = X_train_balanced[train_idx], y_train_balanced[train_idx]
        X_fold_val,   y_fold_val   = X_train_balanced[val_idx],   y_train_balanced[val_idx]
        knn.fit(X_fold_train, y_fold_train)
        y_fold_pred = knn.predict(X_fold_val)
        fold_f1.append(f1_score(y_fold_val, y_fold_pred, average='weighted', zero_division=0))
    f1_scores.append(np.mean(fold_f1))

k_optimo_idx = np.argmax(f1_scores)
k_optimo = list(k_range)[k_optimo_idx]

plt.figure(figsize=(10, 5))
plt.plot(k_range, f1_scores, marker='o', linewidth=2, markersize=8)
plt.axvline(x=k_optimo, color='red', linestyle='--', label=f'k óptimo = {k_optimo}')
plt.title('Búsqueda de k Óptimo: F1 medio CV (5 folds) — solo train set')
plt.xlabel('Número de vecinos (k)')
plt.ylabel('F1-Score Ponderado (media CV)')
plt.xticks(k_range)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ k óptimo encontrado: {k_optimo}")
print(f"  F1-Score medio CV (train): {max(f1_scores):.4f}")
print(f"  X_test no fue utilizado en este paso.")


In [ ]:
# ===== PASO 5: ENTRENAMIENTO FINAL CON k ÓPTIMO =====
# Entrenar SOLO con datos balanceados de TRAIN
knn_final = KNeighborsClassifier(n_neighbors=k_optimo, metric='cosine')
knn_final.fit(X_train_balanced, y_train_balanced)

print(f"✓ Modelo KNN entrenado con k={k_optimo}")
print(f"  Datos de entrenamiento: {X_train_balanced.shape[0]} muestras")

# Clasificar los tickets no clasificados
y_pred_no_clasificados = knn_final.predict(X_no_clasificados)
df_no_clasificados['Clasificación_predicha'] = y_pred_no_clasificados

print(f"\n✓ Clasificados {df_no_clasificados.shape[0]} tickets sin clasificar")
print('\nEjemplos de tickets no clasificados y su predicción:')
display(df_no_clasificados[['texto_unificado', 'Clasificación_predicha']].head(10))

In [ ]:
# ===== PASO 6: EVALUACIÓN EN TEST SET (sin data leakage) =====
# Predecir en TEST SET (datos que el modelo NUNCA vio)
y_pred_test = knn_final.predict(X_test)

# Calcular métricas robustas
accuracy = accuracy_score(y_test, y_pred_test)
precision_weighted = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
recall_weighted = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)
f1_weighted = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)

print("\n" + "="*60)
print("EVALUACIÓN EN TEST SET (Datos nuevos)")
print("="*60)
print(f"Accuracy:              {accuracy:.4f}")
print(f"Precision (ponderado): {precision_weighted:.4f}")
print(f"Recall (ponderado):    {recall_weighted:.4f}")
print(f"F1-Score (ponderado):  {f1_weighted:.4f}")

print("="*60)
print("REPORTE DE CLASIFICACIÓN (Por clase):")
print(classification_report(y_test, y_pred_test, zero_division=0))


In [ ]:
# Visualizar matriz de confusión normalizada
labels = sorted(list(set(list(y_test) + list(y_pred_test))))
cm = confusion_matrix(y_test, y_pred_test)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=labels, yticklabels=labels, cbar_kws={'label': 'Proporción'})
plt.xlabel('Predicción')
plt.ylabel('Etiqueta Real')
plt.title('Matriz de Confusión Normalizada (Test Set) - Proporciones por fila')
plt.tight_layout()
plt.show()


## Comparativa de Clasificadores

### Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf_model.fit(X_train_balanced, y_train_balanced)

print("✓ Random Forest entrenado  (n_estimators=200)")


### Logistic Regression

In [ ]:
lr_model = LogisticRegression(C=1.0, max_iter=3000, random_state=42, solver='lbfgs')
lr_model.fit(X_train_balanced, y_train_balanced)

print("✓ Logistic Regression entrenada  (C=1.0)")


In [ ]:

# ── Combinar los tres modelos y evaluar en test set ────────────────────────
modelos = {
    'KNN':                 knn_final,
    'Random Forest':       rf_model,
    'Logistic Regression': lr_model,
}

resultados = []
predicciones_modelos = {}
for nombre, modelo in modelos.items():
    y_pred_m = modelo.predict(X_test)
    predicciones_modelos[nombre] = y_pred_m
    resultados.append({
        'Modelo':    nombre,
        'Accuracy':  accuracy_score(y_test, y_pred_m),
        'F1-Score':  f1_score(y_test, y_pred_m, average='weighted', zero_division=0),
        'Precision': precision_score(y_test, y_pred_m, average='weighted', zero_division=0),
        'Recall':    recall_score(y_test, y_pred_m, average='weighted', zero_division=0)
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')

print("=== Métricas comparativas (conjunto de prueba) ===\n")
print(df_resultados.round(4).to_string())

mejor = df_resultados['F1-Score'].idxmax()
print(f"\n► Mejor clasificador por F1-Score (weighted): {mejor}  "
      f"({df_resultados.loc[mejor, 'F1-Score']:.4f})")


In [ ]:
metricas = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
x = np.arange(len(metricas))
n_modelos = len(df_resultados)
ancho = 0.18
colores = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (nombre, row) in enumerate(df_resultados.iterrows()):
    valores = [row[m] for m in metricas]
    barras = ax.bar(x + i * ancho, valores, ancho, label=nombre, color=colores[i], alpha=0.88)
    for barra, val in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width() / 2,
                barra.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

offset_centro = (n_modelos - 1) * ancho / 2
ax.set_xticks(x + offset_centro)
ax.set_xticklabels(metricas, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Comparativa de Clasificadores — conjunto de prueba', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
labels = sorted(list(set(list(y_test))))
fig, axes = plt.subplots(1, 3, figsize=(36, 11))

for ax, (nombre, y_pred_m) in zip(axes, predicciones_modelos.items()):
    cm = confusion_matrix(y_test, y_pred_m, labels=labels)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2%', cmap='Blues',
        xticklabels=labels, yticklabels=labels,
        cbar_kws={'label': 'Proporción'}, ax=ax
    )
    ax.set_xlabel('Predicción', fontsize=11)
    ax.set_ylabel('Etiqueta Real', fontsize=11)
    ax.set_title(f'Matriz de Confusión Normalizada\n{nombre}', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


## Análisis de Errores

### Detalle de errores — Logistic Regression

In [ ]:
# ── Análisis detallado de errores — Logistic Regression ────────────────────

# Predecir en test set con Logistic Regression
y_pred_lr = lr_model.predict(X_test)

# Identificar muestras mal clasificadas
indices_errores = np.where(y_test != y_pred_lr)[0]
df_errores = df_test_base.iloc[indices_errores].copy()
df_errores['y_predicho'] = y_pred_lr[indices_errores]
df_errores['es_error'] = True

print(f"Total de errores en test set: {len(df_errores)} de {len(y_test)}")
print(f"Tasa de error: {len(df_errores) / len(y_test) * 100:.2f}%")
print(f"Accuracy: {1 - len(df_errores) / len(y_test):.4f}\n")

# Matriz de confusión de errores
print("Top 15 confusiones más frecuentes:")
confusiones = df_errores.groupby(['y_real', 'y_predicho']).size().reset_index(name='count')
confusiones = confusiones.sort_values('count', ascending=False).head(15)
print(confusiones.to_string(index=False))

# Mostrar ejemplos de errores por tipo
print("\n" + "="*80)
print("EJEMPLOS DE ERRORES (primeras 10 muestras mal clasificadas):")
print("="*80)

cols_error = [c for c in ['number', 'short_description', 'texto_unificado', 'y_real', 'y_predicho'] 
              if c in df_errores.columns]
with pd.option_context('display.max_colwidth', 100, 'display.max_rows', None):
    display(df_errores[cols_error].head(10))

# Gráfico: clases más confundidas
fig, ax = plt.subplots(figsize=(12, 6))
confusiones_top = confusiones.head(10)
barras = ax.barh(range(len(confusiones_top)), confusiones_top['count'], color='coral', alpha=0.8)
ax.set_yticks(range(len(confusiones_top)))
labels_y = [f"{row['y_real']} → {row['y_predicho']}" 
            for _, row in confusiones_top.iterrows()]
ax.set_yticklabels(labels_y)
ax.set_xlabel('Número de errores', fontsize=11)
ax.set_title('Top 10 Confusiones — Logistic Regression', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for barra, val in zip(barras, confusiones_top['count']):
    ax.text(val + 0.1, barra.get_y() + barra.get_height()/2, str(int(val)), 
            va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Tickets que presentan confusión (errores de clasificación en test)
# Usa df_errores generado en la celda de análisis de errores.

if 'df_errores' not in globals():
    raise ValueError("No existe 'df_errores'. Ejecuta primero la celda de análisis de errores.")

# Crear identificador de ticket si no existe la columna 'number'
if 'number' not in df_errores.columns:
    df_errores = df_errores.copy()
    df_errores['number'] = df_errores.index.astype(str)

print(f"Total de tickets con confusión: {len(df_errores)}")

# 1) Resumen por tipo de confusión (real -> predicho) con IDs de tickets
resumen_confusiones_tickets = (
    df_errores.groupby(['y_real', 'y_predicho'], as_index=False)
    .agg(
        total_tickets=('number', 'count'),
        tickets=('number', lambda x: ', '.join(x.astype(str).head(30)))
    )
    .sort_values('total_tickets', ascending=False)
)

print("\nTop confusiones con tickets asociados:")
display(resumen_confusiones_tickets)


In [ ]:
# Exportar a CSV el resumen y/o detalle de tickets con confusión

fecha_hoy = pd.Timestamp.today().strftime("%Y-%m-%d")
carpeta_output = 'Resultados'
os.makedirs(carpeta_output, exist_ok=True)

# 1) Resumen de confusiones (si existe)
resumen_confusiones = globals().get('resumen_confusiones_tickets')
if resumen_confusiones is not None:
    ruta_resumen = os.path.join(carpeta_output, f"resumen_confusiones_tickets_{fecha_hoy}.csv")
    resumen_confusiones.to_csv(ruta_resumen, index=False, encoding='utf-8-sig')
    print(f"✓ Resumen exportado: {ruta_resumen} ({len(resumen_confusiones)} filas)")

# 2) Detalle de tickets confundidos (si existe)
tickets_conf = globals().get('tickets_confundidos')
if tickets_conf is not None:
    ruta_detalle = os.path.join(carpeta_output, f"tickets_confundidos_{fecha_hoy}.csv")
    tickets_conf.to_csv(ruta_detalle, index=False, encoding='utf-8-sig')
    print(f"✓ Detalle exportado: {ruta_detalle} ({len(tickets_conf)} filas)")
elif 'df_errores' in globals():
    cols_detalle = [c for c in ['number', 'short_description', 'texto_unificado', 'y_real', 'y_predicho'] if c in df_errores.columns]
    detalle_tmp = df_errores[cols_detalle].sort_values(['y_real', 'y_predicho', 'number']).reset_index(drop=True)
    ruta_detalle = os.path.join(carpeta_output, f"tickets_confundidos_{fecha_hoy}.csv")
    detalle_tmp.to_csv(ruta_detalle, index=False, encoding='utf-8-sig')
    print(f"✓ Detalle exportado (desde df_errores): {ruta_detalle} ({len(detalle_tmp)} filas)")

## Generación de csv tickets ya clasificados tras algoritmo

In [ ]:

# ── Usar el mejor clasificador para predecir tickets no clasificados ──────────
mejor_modelo = modelos[mejor]
y_pred_final = mejor_modelo.predict(X_no_clasificados)

# ── Obtener confianza de la predicción ────────────────────────────────────────
# NOTA: LinearSVC no tiene predict_proba → se usa decision_function + softmax,
#       pero esos valores NO son probabilidades calibradas (son distancias al hiperplano).
#       Por eso se usa un umbral más bajo (0.10) cuando el modelo es LinearSVC.
if hasattr(mejor_modelo, 'predict_proba'):
    proba_matrix = mejor_modelo.predict_proba(X_no_clasificados)
    confianza = proba_matrix.max(axis=1)
    umbral_revision = 0.40
    tipo_confianza = 'probabilidad calibrada'
else:
    # Softmax sobre decision_function — no es probabilidad real, solo aproximación relativa
    scores = mejor_modelo.decision_function(X_no_clasificados)
    exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
    proba_matrix = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    confianza = proba_matrix.max(axis=1)
    umbral_revision = 0.10   # umbral ajustado para scores softmax de LinearSVC
    tipo_confianza = 'softmax(decision_function) — no calibrada'

# ── Construir DataFrame resultado ─────────────────────────────────────────────
cols_visibles = [c for c in ['number', 'short_description', 'description',
                              'u_subcategory', 'u_subcategory_2', 'texto_unificado',
                              'Clasificación']
                 if c in df_no_clasificados.columns]

df_resultado = df_no_clasificados[cols_visibles].copy()
df_resultado['Clasificación_predicha'] = y_pred_final
df_resultado['Confianza'] = np.round(confianza, 4)
df_resultado['Requiere_revision'] = confianza < umbral_revision
df_resultado['Modelo_usado'] = mejor

print(f"Modelo usado: {mejor}  |  Tipo de confianza: {tipo_confianza}")
print(f"Umbral de revisión aplicado: {umbral_revision}")
print(f"\nTotal de tickets clasificados: {len(df_resultado)}")
print(f"\nDistribución de predicciones:")
print(df_resultado['Clasificación_predicha'].value_counts().to_string())
print(f"\nTickets con baja confianza (< {umbral_revision}): {df_resultado['Requiere_revision'].sum()} "
      f"({df_resultado['Requiere_revision'].mean():.1%} del total)")


In [ ]:
display(df_resultado)

In [ ]:
from datetime import datetime

# Carpeta output/ al mismo nivel que Data/ (un nivel arriba de Analisis/)
fecha_hoy = datetime.now().strftime("%Y-%m-%d")
nombre_modelo_csv = mejor.replace(" ", "_")
nombre_archivo = f"tickets_clasificados_{nombre_modelo_csv}_{fecha_hoy}.csv"

carpeta_output = 'Resultados'
os.makedirs(carpeta_output, exist_ok=True)

ruta_salida = os.path.normpath(os.path.join(carpeta_output, nombre_archivo))
df_resultado.to_csv(ruta_salida, index=False, encoding='utf-8-sig', sep=';')

print(f"✓ CSV exportado: {ruta_salida}")
print(f"  Filas: {len(df_resultado)}  |  Columnas: {list(df_resultado.columns)}")
print(f"\n  Alta confianza (≥ {umbral_revision}): {(~df_resultado['Requiere_revision']).sum()} tickets")
print(f"  Requiere revisión  (< {umbral_revision}): {df_resultado['Requiere_revision'].sum()} tickets")

# ── Histograma de confianza ───────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.hist(df_resultado['Confianza'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(umbral_revision, color='red', linestyle='--', linewidth=1.5,
            label=f'Umbral revisión ({umbral_revision})')
plt.xlabel('Confianza de la predicción')
plt.ylabel('Número de tickets')
plt.title(f'Distribución de confianza — {mejor}')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Combinar tickets clasificados originales + tickets reclasificados ──────────

# 1. DataFrame de tickets clasificados (originales, sin cambios)
df_clasificados_final = df_clasificados[['number', 'short_description', 'description',
                                          'u_subcategory', 'u_subcategory_2', 
                                          'texto_unificado', 'Clasificación']].copy()
df_clasificados_final['Fuente'] = 'Original_Clasificado'
df_clasificados_final['Confianza'] = 1.0

# 2. DataFrame de tickets reclasificados (que eran sin_clasificar)
df_reclasificados = df_no_clasificados[['number', 'short_description', 'description',
                                         'u_subcategory', 'u_subcategory_2',
                                         'texto_unificado']].copy()
df_reclasificados['Clasificación'] = y_pred_final
df_reclasificados['Fuente'] = 'Reclasificado_IA'
df_reclasificados['Confianza'] = confianza

# 3. Combinar ambos DataFrames
df_completo = pd.concat([df_clasificados_final, df_reclasificados], 
                         ignore_index=True, sort=False)

# 4. Ordenar por número de ticket
df_completo = df_completo.sort_values('number').reset_index(drop=True)

# 5. Exportar CSV completo
fecha_hoy = datetime.now().strftime("%Y-%m-%d")
nombre_archivo_completo = f"tickets_clasificados_completo_{mejor.replace(' ', '_')}_{fecha_hoy}.csv"
carpeta_output = 'Resultados'
os.makedirs(carpeta_output, exist_ok=True)

ruta_salida_completa = os.path.normpath(os.path.join(carpeta_output, nombre_archivo_completo))
df_completo.to_csv(ruta_salida_completa, index=False, encoding='utf-8-sig')

print(f"✓ CSV COMPLETO exportado: {ruta_salida_completa}")
print(f"  Total de tickets: {len(df_completo)}")
print(f"  - Clasificados originales: {len(df_clasificados_final)}")
print(f"  - Reclasificados (sin_clasificar → IA): {len(df_reclasificados)}")
print(f"\nDistribución final de clasificaciones:") 
print(df_completo['Clasificación'].value_counts().to_string())
print(f"\nColumnas del CSV: {list(df_completo.columns)}")

In [ ]:
# ── Persistencia del modelo y vectorizer ─────────────────────────────────────
# Se guardan juntos en la carpeta 'modelo/' para garantizar que la
# transformación TF-IDF sea idéntica al momento de inferencia futura.
nombre_modelo = mejor.replace(' ', '_')

carpeta_modelo = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'semisupervised_model'))
os.makedirs(carpeta_modelo, exist_ok=True)

ruta_modelo     = os.path.join(carpeta_modelo, f'modelo_{nombre_modelo}.joblib')
ruta_vectorizer = os.path.join(carpeta_modelo, 'vectorizer_tfidf.joblib')

joblib.dump(mejor_modelo, ruta_modelo)
joblib.dump(vectorizer,   ruta_vectorizer)

print(f"✓ Modelo guardado:     {ruta_modelo}")
print(f"✓ Vectorizer guardado: {ruta_vectorizer}")
print()
print("Para cargar y usar en otro script:")
print(f"  modelo     = joblib.load('modelo/modelo_{nombre_modelo}.joblib')")
print(f"  vectorizer = joblib.load('modelo/vectorizer_tfidf.joblib')")
print(f"  X_nuevo    = vectorizer.transform(textos_nuevos)")
print(f"  prediccion = modelo.predict(X_nuevo)")
